# 🗂️ Git & GitHub for Data Science

> **Target outcome:** Stop zipping files and start working like a professional.

---

## Why This Matters

Every data scientist eventually hits the same wall. Their project folder looks like this:

```
📁 my_project/
   model_v1.ipynb
   model_v2.ipynb
   model_v2_FINAL.ipynb
   model_v2_FINAL_fixed.ipynb
   model_v2_REALLY_FINAL.ipynb
   model_v2_REALLY_FINAL_FORREAL.ipynb
```

This is version control without a system — and it's a nightmare. **Git** solves this by being three things at once:

| Git is a... | What that means |
|-------------|----------------|
| **Time machine** | You can travel back to any previous state of your project |
| **Collaboration contract** | Everyone on a team agrees on what the "real" code is |
| **Safety net** | You can experiment freely, knowing you can always undo |

---

## 🗺️ What We'll Cover

| # | Topic |
|---|-------|
| 1 | How Git thinks: the mental model |
| 2 | The local workflow: init, add, commit, log |
| 3 | GitHub: pushing to the cloud |
| 4 | Branching & merging for experimentation |
| 5 | Merge conflicts: what they are and how to fix them |
| 6 | Collaboration: pull requests & code review |
| 7 | Best practices: commits, .gitignore, README |
| 8 | Hands-on lab simulation |

---

> **Note on running this notebook:** Git commands are shell commands, not Python. Throughout this notebook we use Python's `subprocess` module and file I/O to *simulate* a real Git workflow inside a temporary directory — so every cell is runnable and produces real output.

In [ ]:
# ============================================================
# SETUP: Create a temporary sandbox directory for our lab
# ============================================================
import subprocess, os, shutil, tempfile
from pathlib import Path

def run(cmd, cwd=None, capture=True):
    """Run a shell command and print output cleanly."""
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=capture, text=True
    )
    out = (result.stdout + result.stderr).strip()
    if out:
        print(out)
    return out

# Create a fresh temporary directory for the entire lab
LAB_DIR = Path(tempfile.mkdtemp()) / 'ml_project'
LAB_DIR.mkdir()

# Configure git identity for the lab (required for commits)
run('git config --global user.email "learner@ai4sid.org"')
run('git config --global user.name "AI4SID Learner"')
run('git config --global init.defaultBranch main')

print(f'Lab sandbox created at: {LAB_DIR}')
print('Git version:', run('git --version'))

---
## 1. How Git Thinks — The Mental Model

Before touching any commands, you need to understand **where your files live** in Git's world. There are three zones:

```
┌─────────────────┐     git add      ┌─────────────────┐    git commit    ┌─────────────────┐
│                 │ ──────────────►  │                 │ ──────────────►  │                 │
│  Working Tree   │                  │  Staging Area   │                  │    Repository   │
│  (your files)   │ ◄──────────────  │  (the loading   │                  │  (the history   │
│                 │   git checkout   │     dock)       │                  │     book)       │
└─────────────────┘                  └─────────────────┘                  └─────────────────┘
```

**Working Tree:** The actual files on your computer. When you edit `model.py`, you're editing the working tree.

**Staging Area (Index):** A loading dock. You *choose* which changes to package into the next snapshot. This is why Git is powerful — you can edit 5 things but only commit 2 of them.

**Repository (.git folder):** The permanent history book. Every commit is a timestamped snapshot with a unique ID (called a SHA hash). This is what you push to GitHub.

---

**What is a commit?** Think of it as a photograph of your entire project at a specific moment in time. Unlike saving a file (which overwrites), committing *adds a new photo to the album* — the old ones stay.

Each commit has:
- A **SHA hash** — a unique 40-character fingerprint (e.g., `a3f5c21`)
- An **author** and **timestamp**
- A **message** describing what changed
- A pointer to its **parent commit** (forming a chain = history)

---
## 2. The Local Workflow

### The 5 Commands You'll Use Every Day

| Command | What it does | When to use it |
|---------|-------------|----------------|
| `git init` | Creates a new repository | Once, at project start |
| `git status` | Shows what's changed | Constantly — your dashboard |
| `git add <file>` | Moves changes to staging | Before committing |
| `git commit -m "msg"` | Snapshots staged changes | After meaningful progress |
| `git log` | Shows commit history | When you need to look back |

In [ ]:
# ============================================================
# STEP 1: git init — Start tracking a project
# ============================================================
print('=== git init ===')
run('git init', cwd=LAB_DIR)

print('\n--- What just happened? ---')
print('A hidden .git/ folder was created. Let\'s see what\'s inside:')
run('ls -la .git/', cwd=LAB_DIR)

print()
print('KEY ITEMS:')
print('  HEAD       → points to the current branch (currently: main)')
print('  objects/   → where all your commits and file snapshots are stored')
print('  config     → repository-level configuration')
print()
print('This .git folder IS your repository. Delete it and you lose all history.')

In [ ]:
# ============================================================
# STEP 2: Create some files — simulate starting a project
# ============================================================
# Create a preprocessing script
(LAB_DIR / 'preprocess.py').write_text("""
import pandas as pd

def load_and_clean(filepath):
    df = pd.read_csv(filepath)
    df = df.dropna()
    df = df.drop_duplicates()
    return df

if __name__ == '__main__':
    df = load_and_clean('data/diabetes.csv')
    print(f'Loaded {len(df)} clean records')
""".strip())

# Create a model training script
(LAB_DIR / 'train.py').write_text("""
from sklearn.linear_model import LogisticRegression
from preprocess import load_and_clean

def train_model(X, y):
    model = LogisticRegression(random_state=42)
    model.fit(X, y)
    return model
""".strip())

# Create a data directory (but NOT the CSV — that'll go in .gitignore)
(LAB_DIR / 'data').mkdir()
(LAB_DIR / 'data' / 'diabetes.csv').write_text('Pregnancies,Glucose,Outcome\n6,148,1\n')

print('Files created in project:')
run('find . -type f | sort', cwd=LAB_DIR)

print()
print('=== git status ===')
run('git status', cwd=LAB_DIR)

### Understanding `git status` output

Git is telling us: *"I see these files exist, but I'm not tracking them yet."*

Files can be in one of four states:

```
Untracked   →  Git has never seen this file
Modified    →  Git knows this file but it changed since the last commit
Staged      →  Marked for inclusion in the next commit
Committed   →  Safely stored in the repository history
```

**Pro tip:** Run `git status` constantly. It's your GPS — it always tells you where you are.

In [ ]:
# ============================================================
# STEP 3: .gitignore — Tell Git what to NEVER track
# ============================================================
# In data science, you should NEVER commit:
# - Large datasets (CSV, parquet, etc.)
# - Model weights (.pkl, .h5, .pt)
# - API keys / credentials
# - __pycache__ folders
# - Jupyter checkpoint folders

gitignore_content = """# Data files — too large and often sensitive
data/
*.csv
*.parquet
*.xlsx

# Trained models — reproducible from code
models/
*.pkl
*.h5
*.pt

# Environment & secrets — NEVER commit these
.env
secrets.json

# Python artifacts
__pycache__/
*.pyc
.ipynb_checkpoints/

# OS files
.DS_Store
Thumbs.db
"""

(LAB_DIR / '.gitignore').write_text(gitignore_content)

print('=== .gitignore created ===')
print(gitignore_content)

print('=== git status (now data/ should be ignored) ===')
run('git status', cwd=LAB_DIR)

In [ ]:
# ============================================================
# STEP 4: git add — Stage files for commit
# ============================================================
print('=== Staging specific files ===')
run('git add preprocess.py .gitignore', cwd=LAB_DIR)
run('git status', cwd=LAB_DIR)

print()
print('Notice: preprocess.py and .gitignore are staged (green).')
print('train.py is still untracked — we can control WHAT goes into each commit.')
print()
print('Common git add patterns:')
print('  git add filename.py      → stage one file')
print('  git add *.py             → stage all Python files')
print('  git add .                → stage EVERYTHING unignored (use carefully!)')
print('  git add -p               → interactive: review each change before staging')

In [ ]:
# ============================================================
# STEP 5: git commit — Save a snapshot
# ============================================================
print('=== First commit ===')
run('git commit -m "Add preprocessing script and gitignore"', cwd=LAB_DIR)

print()
print('=== Now add train.py in a separate commit (good habit!) ===')
run('git add train.py', cwd=LAB_DIR)
run('git commit -m "Add initial model training script"', cwd=LAB_DIR)

print()
print('WHY separate commits?')
print('  → Each commit is a rollback point. Smaller commits = more precise control.')
print('  → If train.py breaks things, you can revert JUST that commit.')
print('  → "git add ." then one big commit = hard to undo specific changes.')

In [ ]:
# ============================================================
# STEP 6: git log — Browse history
# ============================================================
# Make a few more commits to build up history
(LAB_DIR / 'evaluate.py').write_text("""
from sklearn.metrics import accuracy_score, classification_report

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    return accuracy_score(y_test, y_pred)
""".strip())
run('git add evaluate.py', cwd=LAB_DIR)
run('git commit -m "Add model evaluation functions"', cwd=LAB_DIR)

(LAB_DIR / 'README.md').write_text('# ML Project\n\nDiabetes prediction model.')
run('git add README.md', cwd=LAB_DIR)
run('git commit -m "Add README"', cwd=LAB_DIR)

print('=== git log — Full history ===')
run('git log', cwd=LAB_DIR)

print()
print('=== git log --oneline — Compact view (most useful day-to-day) ===')
run('git log --oneline', cwd=LAB_DIR)

print()
print('=== git log --oneline --graph — Visualise branches ===')
run('git log --oneline --graph --all', cwd=LAB_DIR)

---
## 3. GitHub — Taking Your Work to the Cloud

Git is **local** — it lives on your machine. GitHub is a **remote** hosting service where you push your repository so it's:
- Backed up (no more losing work if your laptop dies)
- Shareable (teammates can clone it)
- Collaborative (multiple people can contribute)

### The Local ↔ Remote Relationship

```
Your Laptop                          GitHub.com
┌─────────────────┐                 ┌─────────────────┐
│  Local Repo     │  ─── push ───►  │  Remote Repo    │
│  (.git folder)  │  ◄─── pull ───  │  (origin)       │
└─────────────────┘                 └─────────────────┘
```

### Key Commands

| Command | What it does |
|---------|-------------|
| `git remote add origin <url>` | Links your local repo to GitHub |
| `git push origin main` | Uploads your commits to GitHub |
| `git pull origin main` | Downloads the latest changes from GitHub |
| `git clone <url>` | Downloads a repo from GitHub to your machine |
| `git fetch` | Checks what's new on remote (without merging) |

### Workflow for an Existing Project

```bash
# 1. Create the repo on GitHub (empty, no README)
# 2. Link local to remote
git remote add origin https://github.com/yourname/ml-project.git

# 3. Push
git push -u origin main   # -u sets 'origin main' as the default

# After that, just:
git push
```

### The Daily Collaboration Loop

```bash
# Start of every work session:
git pull            # Get teammates' latest changes

# ... do your work ...

git add <files>
git commit -m "Your message"
git push            # Share your work
```

> ⚠️ **Always pull before you push.** If a teammate pushed while you were working, Git needs to merge their changes with yours first.

In [ ]:
# ============================================================
# SIMULATE: Remote operations (without needing GitHub account)
# We create a 'bare' repo to act as the remote
# ============================================================
import tempfile

# Create a fake 'remote' (like GitHub)
REMOTE_DIR = Path(tempfile.mkdtemp()) / 'remote_repo.git'
REMOTE_DIR.mkdir()
run('git init --bare', cwd=REMOTE_DIR)

print('=== Simulated GitHub remote created ===')
print(f'Remote path: {REMOTE_DIR}')
print()

# Link local repo to the remote
print('=== git remote add origin ===')
run(f'git remote add origin {REMOTE_DIR}', cwd=LAB_DIR)
run('git remote -v', cwd=LAB_DIR)

print()
print('=== git push ===')
run('git push -u origin main', cwd=LAB_DIR)

print()
print('✅ All local commits are now on the remote.')
print('   In a real workflow, "remote" = GitHub/GitLab/Bitbucket URL.')

In [ ]:
# ============================================================
# SIMULATE: A teammate clones and makes a change
# ============================================================
TEAMMATE_DIR = Path(tempfile.mkdtemp()) / 'teammate_ml_project'

print('=== Teammate clones the repo ===')
run(f'git clone {REMOTE_DIR} {TEAMMATE_DIR}')

# Teammate makes a change
run('git config user.name "Teammate Amara"', cwd=TEAMMATE_DIR)
run('git config user.email "amara@ai4sid.org"', cwd=TEAMMATE_DIR)
(TEAMMATE_DIR / 'requirements.txt').write_text('scikit-learn>=1.3\npandas>=2.0\nnumpy>=1.24\n')
run('git add requirements.txt', cwd=TEAMMATE_DIR)
run('git commit -m "Add project requirements file"', cwd=TEAMMATE_DIR)
run('git push origin main', cwd=TEAMMATE_DIR)

print()
print('=== You pull teammate\'s changes ===')
run('git pull origin main', cwd=LAB_DIR)

print()
print('=== Full project log now shows both contributors ===')
run('git log --oneline', cwd=LAB_DIR)

print()
print('The requirements.txt from Amara is now in your local repo:')
run('cat requirements.txt', cwd=LAB_DIR)

---
## 4. Branching & Merging — Experimenting Without Fear

Branches are Git's most powerful feature for data science. They let you:
- Try a new model architecture without touching the working code
- Let multiple people develop different features simultaneously
- Keep `main` always clean and deployable

### The Branch Model

```
main ──●──────────────────────────────●── (always stable)
        \                            /
feature  ●──●──●──●  (experiment)  ●
              \
bugfix          ●──●  (quick fix)
```

### Naming Conventions

| Branch Type | Example Names |
|-------------|---------------|
| Feature | `feature/xgboost-experiment` |
| Bugfix | `fix/insulin-imputation-bug` |
| Experiment | `exp/smote-oversampling` |
| Release | `release/v1.0` |

### Key Commands

```bash
git branch                        # list all branches
git checkout -b feature/new-exp   # create AND switch to a new branch
git checkout main                 # switch back to main
git merge feature/new-exp         # merge a branch into current branch
git branch -d feature/new-exp     # delete a branch (after merging)
```

In [ ]:
# ============================================================
# BRANCHING: Create a feature branch for a model experiment
# ============================================================
print('=== Current branches ===')
run('git branch', cwd=LAB_DIR)

print()
print('=== Create and switch to a new experiment branch ===')
run('git checkout -b feature/random-forest-experiment', cwd=LAB_DIR)
run('git branch', cwd=LAB_DIR)

# Make changes on this branch
(LAB_DIR / 'train_rf.py').write_text("""
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

def train_random_forest(X, y, n_estimators=100, max_depth=10):
    """Train a Random Forest and evaluate with cross-validation."""
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
    print(f'Random Forest CV AUC: {np.mean(scores):.3f} ± {np.std(scores):.3f}')
    model.fit(X, y)
    return model
""".strip())

run('git add train_rf.py', cwd=LAB_DIR)
run('git commit -m "Add Random Forest training script with CV evaluation"', cwd=LAB_DIR)

# More commits on the branch
(LAB_DIR / 'train_rf.py').write_text(
    (LAB_DIR / 'train_rf.py').read_text() + 
    '\n\ndef get_feature_importance(model, feature_names):\n    return dict(zip(feature_names, model.feature_importances_))\n'
)
run('git add train_rf.py', cwd=LAB_DIR)
run('git commit -m "Add feature importance extraction to RF model"', cwd=LAB_DIR)

print()
print('=== Branch history ===')
run('git log --oneline --graph --all', cwd=LAB_DIR)

In [ ]:
# ============================================================
# MERGING: Bring the experiment back into main
# ============================================================
print('=== Switch back to main ===')
run('git checkout main', cwd=LAB_DIR)

print()
print('=== Check: train_rf.py does NOT exist on main yet ===')
run('ls', cwd=LAB_DIR)

print()
print('=== Merge the experiment branch into main ===')
run('git merge feature/random-forest-experiment --no-ff -m "Merge RF experiment: CV AUC validated"', cwd=LAB_DIR)

print()
print('=== Now train_rf.py IS on main ===')
run('ls', cwd=LAB_DIR)

print()
print('=== Clean up: delete the feature branch ===')
run('git branch -d feature/random-forest-experiment', cwd=LAB_DIR)

print()
print('=== Final history showing the merge ===')
run('git log --oneline --graph', cwd=LAB_DIR)

---
## 5. Merge Conflicts — What They Are and How to Fix Them

A **merge conflict** happens when two branches both changed the *same lines* of the *same file*. Git doesn't know which version to keep — so it asks you.

This is **one of the most disorienting experiences** for new Git users. Understanding what's happening makes it manageable.

### What a Conflict Looks Like

When a conflict occurs, Git marks the file like this:

```python
<<<<<<< HEAD                          ← your current branch's version
model = LogisticRegression(C=1.0)
=======                               ← separator
model = LogisticRegression(C=0.1)    ← the incoming branch's version
>>>>>>> feature/hyperparameter-tune
```

**You must:**
1. Open the file
2. Decide which version to keep (or write a combined version)
3. Delete ALL the `<<<<<<<`, `=======`, `>>>>>>>` markers
4. `git add` the resolved file
5. `git commit` to complete the merge

In [ ]:
# ============================================================
# SIMULATE A MERGE CONFLICT and resolve it
# ============================================================
import re

# Branch A: changes the model hyperparameters
run('git checkout -b feature/tune-regularization', cwd=LAB_DIR)
train_content_a = """
from sklearn.linear_model import LogisticRegression
from preprocess import load_and_clean

def train_model(X, y):
    # Branch A: tried lower regularization (C=10.0)
    model = LogisticRegression(C=10.0, max_iter=1000, random_state=42)
    model.fit(X, y)
    return model
""".strip()
(LAB_DIR / 'train.py').write_text(train_content_a)
run('git add train.py', cwd=LAB_DIR)
run('git commit -m "Tune: increase C to 10.0 for less regularization"', cwd=LAB_DIR)
run('git checkout main', cwd=LAB_DIR)

# Branch B (main): also changed the same file differently
train_content_b = """
from sklearn.linear_model import LogisticRegression
from preprocess import load_and_clean

def train_model(X, y):
    # Main branch: added solver='saga' for large datasets
    model = LogisticRegression(solver='saga', max_iter=500, random_state=42)
    model.fit(X, y)
    return model
""".strip()
(LAB_DIR / 'train.py').write_text(train_content_b)
run('git add train.py', cwd=LAB_DIR)
run('git commit -m "Switch to saga solver for scalability"', cwd=LAB_DIR)

print('=== Attempting merge — this will conflict ===')
run('git merge feature/tune-regularization', cwd=LAB_DIR)

print()
print('=== Conflict! Git cannot decide — here\'s what the file looks like ===')
conflicted = (LAB_DIR / 'train.py').read_text()
print(conflicted)

print()
print('=== RESOLVING: We keep BOTH changes (C=10.0 AND solver=saga) ===')
resolved = """
from sklearn.linear_model import LogisticRegression
from preprocess import load_and_clean

def train_model(X, y):
    # RESOLVED: Combined both improvements
    model = LogisticRegression(C=10.0, solver='saga', max_iter=1000, random_state=42)
    model.fit(X, y)
    return model
""".strip()
(LAB_DIR / 'train.py').write_text(resolved)

run('git add train.py', cwd=LAB_DIR)
run('git commit -m "Resolve conflict: combine C=10 and saga solver"', cwd=LAB_DIR)
print()
print('✅ Conflict resolved and committed!')
run('git log --oneline --graph', cwd=LAB_DIR)

---
## 6. Collaboration: Pull Requests & Code Review

A **Pull Request (PR)** is a GitHub feature (not a Git command) — it's a *formal request* to merge your branch into another branch, with a built-in review process.

### The Pull Request Workflow

```
1. Fork or clone the repo
2. Create a feature branch
3. Make commits
4. Push branch to GitHub
5. Open a Pull Request on GitHub
6. Teammates review, comment, request changes
7. You update based on feedback
8. PR is approved → merged into main
9. Delete the feature branch
```

### What Makes a Good Pull Request?

| ✅ Good PR | ❌ Bad PR |
|-----------|----------|
| Small and focused (one feature) | Changes 20 files across 5 features |
| Clear title: "Add Random Forest with CV evaluation" | Vague title: "update" |
| Describes *why*, not just *what* | No description |
| Includes test results | No evidence it works |
| Links to issue it closes | No context |

### The GitHub Flow (Most Common for Data Science Teams)

```bash
# 1. Always branch from main
git checkout main
git pull
git checkout -b feature/my-experiment

# 2. Work on your branch
git add . && git commit -m "..."

# 3. Push and open PR
git push origin feature/my-experiment
# → Go to GitHub → 'Compare & Pull Request'

# 4. After PR is merged, clean up
git checkout main
git pull
git branch -d feature/my-experiment
```

---
## 7. Best Practices

### ✍️ Writing Great Commit Messages

A commit message is a letter to your future self (and teammates). Make it count.

**Format:**
```
<type>: <short summary in imperative mood>  (max 72 characters)

<optional body: what changed and WHY — not how>
```

**Types:** `feat:` `fix:` `refactor:` `docs:` `test:` `chore:`

| ❌ Bad Message | ✅ Good Message |
|--------------|----------------|
| `update` | `feat: Add SMOTE oversampling to handle class imbalance` |
| `fix bug` | `fix: Correct insulin imputation — was using mean on skewed data` |
| `I added some stuff` | `refactor: Extract feature engineering into separate module` |
| `final final version` | `feat: Finalize logistic regression baseline (AUC: 0.82)` |

### 📁 The Data Science Project Structure

```
ml-project/
├── README.md           ← Project overview, setup, results
├── requirements.txt    ← Python dependencies
├── .gitignore          ← Ignore data/, models/, .env
├── data/               ← Raw data (GITIGNORED)
│   └── raw/
├── notebooks/          ← Exploratory notebooks
│   └── 01_eda.ipynb
├── src/                ← Production-quality Python modules
│   ├── preprocess.py
│   ├── train.py
│   └── evaluate.py
├── models/             ← Saved model files (GITIGNORED)
└── reports/            ← Figures, results markdown
```

### 📖 The Perfect README

Your README is the *storefront* of your project. A stranger should be able to understand and run your project from the README alone.

```markdown
# Diabetes Risk Prediction

Predicts diabetes onset from clinical measurements using logistic regression
and random forest classifiers. Built for the AI4SID cohort project.

## Key Results
- Best model: Random Forest — AUC 0.847 on held-out test set
- Glucose and BMI are the strongest predictors

## Setup
pip install -r requirements.txt

## Data
Pima Indians Diabetes Dataset (Kaggle). Place in data/raw/diabetes.csv

## How to Run
python src/preprocess.py   # clean data
python src/train.py        # train model
python src/evaluate.py     # evaluate and save results

## Team
- Amara (EDA & preprocessing)
- Kofi (model training)
```

In [ ]:
# ============================================================
# USEFUL COMMANDS: The Git toolkit for daily use
# ============================================================

commands = {
    'INSPECTION': {
        'git status':                      'What changed since last commit?',
        'git log --oneline':               'Show compact commit history',
        'git log --oneline --graph --all': 'Visualise all branches',
        'git diff':                        'Show unstaged changes line by line',
        'git diff --staged':               'Show staged changes before committing',
        'git show <commit>':               'Show what a specific commit changed',
        'git blame <file>':                'See who changed each line of a file',
    },
    'UNDOING MISTAKES': {
        'git restore <file>':              'Discard unstaged changes to a file',
        'git restore --staged <file>':     'Unstage a file (undo git add)',
        'git revert <commit>':             'Create a new commit that undoes a past commit (safe)',
        'git reset --hard HEAD~1':         '⚠️  Delete last commit AND changes (dangerous!)',
        'git reset --soft HEAD~1':         'Undo last commit but KEEP the changes staged',
    },
    'STASHING (save work temporarily)': {
        'git stash':                       'Temporarily shelve your changes',
        'git stash pop':                   'Restore stashed changes',
        'git stash list':                  'See all stashes',
    },
    'WORKING WITH REMOTES': {
        'git remote -v':                   'List all remotes',
        'git fetch':                       'Download remote changes (without merging)',
        'git pull':                        'Download AND merge remote changes',
        'git push':                        'Upload your commits',
        'git push --force-with-lease':     'Force push (safer than --force)',
    }
}

for category, cmds in commands.items():
    print(f'\n{'='*60}')
    print(f'  {category}')
    print('='*60)
    for cmd, desc in cmds.items():
        print(f'  {cmd:<40} {desc}')

---
## 8. 🧪 Hands-On Lab

Complete these steps in order on your own machine:

### Lab A — Solo Workflow (15 min)

```bash
# 1. Create a project folder and initialise Git
mkdir my_ml_project && cd my_ml_project
git init

# 2. Create a .gitignore
echo "data/\n*.csv\n*.pkl\n__pycache__/" > .gitignore

# 3. Create a Python file, stage, and commit
touch preprocess.py
git add .gitignore preprocess.py
git commit -m "feat: Add preprocessing script and gitignore"

# 4. Create a branch for an experiment
git checkout -b feature/try-xgboost
touch train_xgb.py
git add train_xgb.py
git commit -m "feat: Add XGBoost training experiment"

# 5. Merge back to main
git checkout main
git merge feature/try-xgboost

# 6. View history
git log --oneline --graph
```

### Lab B — Team Collaboration (20 min)

Work in pairs. One person is **Owner**, one is **Contributor**.

```bash
# OWNER:
# 1. Create repo on GitHub (don't add README)
# 2. Push your local project:
git remote add origin https://github.com/<you>/my_ml_project.git
git push -u origin main
# 3. Go to Settings → Collaborators → Add your partner

# CONTRIBUTOR:
# 4. Clone the repo
git clone https://github.com/<owner>/my_ml_project.git
cd my_ml_project
# 5. Create a feature branch
git checkout -b feature/add-evaluation-metrics
# 6. Add a file, commit, push
touch evaluate.py
git add evaluate.py
git commit -m "feat: Add model evaluation module"
git push origin feature/add-evaluation-metrics
# 7. Go to GitHub → Open a Pull Request

# OWNER:
# 8. Review the PR on GitHub, leave a comment, then Merge it
# 9. Pull the merged changes locally
git pull
```

---

## ✅ Summary Cheat Sheet

| Task | Command(s) |
|------|------------|
| Start a project | `git init` |
| Check what's changed | `git status` |
| Stage changes | `git add <file>` or `git add .` |
| Save a snapshot | `git commit -m "message"` |
| View history | `git log --oneline --graph` |
| Create a branch | `git checkout -b branch-name` |
| Switch branches | `git checkout branch-name` |
| Merge a branch | `git checkout main && git merge branch-name` |
| Connect to GitHub | `git remote add origin <url>` |
| Upload changes | `git push` |
| Download changes | `git pull` |
| Undo staged changes | `git restore --staged <file>` |
| Undo file edits | `git restore <file>` |

---
*AI4SID — Git & GitHub for Data Science*